# Lab 4 — Part 2: Metrics (CER/UTMOS/SECS), Validation sets, Sampling ablation & RLHF (DPO)

In [1]:
import sys
%pip install -q pytorch-lightning wandb jiwer transformers torchaudio soundfile librosa \
    huggingface-hub safetensors torchcodec resemblyzer
%pip install -q "focalcodec@git+https://github.com/lucadellalib/focalcodec.git@main#egg=focalcodec"

import importlib
for pkg in ["pytorch_lightning","wandb","jiwer","transformers","torchaudio","soundfile",
            "librosa","resemblyzer","focalcodec"]:
    try:
        importlib.import_module(pkg); print(f"  ok  {pkg}")
    except ImportError as e:
        print(f" FAIL {pkg}: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 87.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.4 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.
  ok  pytorch_lightning
  ok  wandb
  ok  jiwer
  ok  transformers
  ok  torchaudio
  ok  soundfile
  ok  librosa


/usr/local/lib/python3.12/dist-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


  ok  resemblyzer
  ok  focalcodec


In [2]:
import os, math, glob, json, warnings, random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import wandb
import jiwer
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, pipeline as hf_pipeline
from focalcodec import FocalCodec
from resemblyzer import VoiceEncoder, preprocess_wav

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

device = cuda


In [3]:
@dataclass
class Config:
    data_dir: Path = Path('/kaggle/input/datasets/mathurinache/the-lj-speech-dataset/LJSpeech-1.1')
    splits_in: Path = Path('/kaggle/input/datasets/kvitkayarish/splits-lab-4')
    ckpt_in:   Path = Path('/kaggle/input/outputs-lab-4-audio')   # khomenkopavlo/outputs-lab-4-audio
    work_dir:  Path = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
    base_model: str = "gpt2"
    max_new_tokens: int = 400
    n_test_per_domain: int = 40      # cap generations per (test-domain) for the ablation
    wandb_project: str = "audioml-lab4-part2"

    @property
    def wavs_dir(self): return self.data_dir / "wavs"
    @property
    def out_audio(self):
        d = self.work_dir / "part2_audio"; d.mkdir(parents=True, exist_ok=True); return d

cfg = Config()

# --- locate the Part-1 checkpoint (Lightning .ckpt). Override CKPT_PATH if auto-search misses. ---
CKPT_PATH = os.environ.get("CKPT_PATH")
if not CKPT_PATH:
    hits = []
    for root in [str(cfg.ckpt_in), str(cfg.work_dir), "/kaggle/input"]:   # your ckpt dataset first
        hits += glob.glob(f"{root}/**/*.ckpt", recursive=True)
    # prefer a "best-*" checkpoint over "last.ckpt"
    hits.sort(key=lambda p: (0 if "best" in os.path.basename(p) else 1, p))
    CKPT_PATH = hits[0] if hits else None
print("CKPT_PATH =", CKPT_PATH)
assert CKPT_PATH, f"No .ckpt found under {cfg.ckpt_in} — set CKPT_PATH manually."

CKPT_PATH = /kaggle/input/datasets/khomenkopavlo/outputs-lab-4-audio/checkpoints/best-04-1915.ckpt


In [4]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
run = wandb.init(project=cfg.wandb_project, name="part2", job_type="eval+dpo")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pavlokhomenko2307 (ai340-nlp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Data splits (same CSVs the Part-1 checkpoint was trained on)

In [5]:
def attach_filepath(df):
    df = df.copy()
    df['filepath'] = df['id'].apply(lambda i: str(cfg.wavs_dir / f"{i}.wav"))
    return df

train_df = attach_filepath(pd.read_csv(cfg.splits_in / "train(1).csv"))
val_df   = attach_filepath(pd.read_csv(cfg.splits_in / "val(1).csv"))
test_df  = attach_filepath(pd.read_csv(cfg.splits_in / "test(1).csv"))
TEXT_COL = 'normalized_transcription'
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print("test domains:", test_df['domain'].value_counts().to_dict() if 'domain' in test_df else "n/a")

train=12241  val=130  test=652
test domains: {'unseen_book': 334, 'in_domain': 318}


## 2. Codec + GPT-TTS model (duplicated from Part 1 so this notebook stands alone)

In [6]:
codec = FocalCodec.from_pretrained('lucadellalib/focalcodec_25hz').to(device).eval()
for p in codec.parameters(): p.requires_grad_(False)
CODEBOOK_SIZE = int(codec.codebook.shape[0])
CODEC_SR_IN, CODEC_SR_OUT = codec.sample_rate_input, codec.sample_rate_output
print(f"codebook={CODEBOOK_SIZE}  sr_in/out={CODEC_SR_IN}/{CODEC_SR_OUT}")

tokenizer = GPT2TokenizerFast.from_pretrained(cfg.base_model)
def enc_text(s): return torch.tensor(tokenizer.encode(str(s), add_special_tokens=False), device=device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/577M [00:00<?, ?B/s]

codebook=8192  sr_in/out=16000/16000


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
@dataclass
class GPT2TTSConfig:
    base_model: str = "gpt2"
    codebook_size: int = 8192
    n_special_tokens: int = 2
    @property
    def audio_vocab_size(self): return self.codebook_size + self.n_special_tokens
    @property
    def bos_id(self): return self.codebook_size
    @property
    def eos_id(self): return self.codebook_size + 1


class GPT2TTS(nn.Module):
    def __init__(self, cfg: GPT2TTSConfig):
        super().__init__()
        self.cfg = cfg
        self.base = GPT2LMHeadModel.from_pretrained(cfg.base_model)
        H, V = self.base.config.n_embd, cfg.audio_vocab_size
        self.audio_emb = nn.Embedding(V, H)
        self.audio_head = nn.Linear(H, V, bias=False)
        nn.init.normal_(self.audio_emb.weight, std=0.02)
        nn.init.normal_(self.audio_head.weight, std=0.02)

    def _build_inputs(self, text_ids, text_lens, audio_ids, audio_lens):
        B, H = text_ids.size(0), self.base.config.n_embd
        dev = text_ids.device
        L = int((text_lens + audio_lens + 2).max())
        inputs = torch.zeros(B, L, H, device=dev, dtype=self.audio_emb.weight.dtype)
        mask   = torch.zeros(B, L, dtype=torch.long, device=dev)
        labels = torch.full((B, L), -100, dtype=torch.long, device=dev)
        wte = self.base.transformer.wte
        bos_emb, eos_emb = self.audio_emb.weight[self.cfg.bos_id], self.audio_emb.weight[self.cfg.eos_id]
        for i in range(B):
            tl, al = int(text_lens[i]), int(audio_lens[i])
            a_ids = audio_ids[i, :al]
            seq = torch.cat([wte(text_ids[i, :tl]), bos_emb[None], self.audio_emb(a_ids), eos_emb[None]], 0)
            n = seq.size(0)
            inputs[i, :n] = seq; mask[i, :n] = 1
            labels[i, tl+1:tl+1+al] = a_ids
            labels[i, tl+1+al] = self.cfg.eos_id
        return inputs, mask, labels

    def forward(self, text_ids, text_lens, audio_ids, audio_lens):
        inputs, mask, labels = self._build_inputs(text_ids, text_lens, audio_ids, audio_lens)
        h = self.base.transformer(inputs_embeds=inputs, attention_mask=mask, return_dict=True).last_hidden_state
        logits = self.audio_head(h).float()
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                               labels[:, 1:].reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits, "labels": labels}

    @torch.no_grad()
    def generate_audio(self, text_ids, max_new_tokens=400, temperature=1.0, top_k=None, top_p=None,
                       repetition_penalty=1.0):
        # top_k=1 == greedy. Set any subset of knobs to compare decoding strategies.
        self.eval()
        dev = text_ids.device
        text_ids = text_ids.view(-1).long().to(dev)
        prefix = torch.cat([self.base.transformer.wte(text_ids).unsqueeze(0),
                            self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)], dim=1)
        mask = torch.ones(1, prefix.size(1), device=dev, dtype=torch.long)
        out = self.base.transformer(inputs_embeds=prefix, attention_mask=mask, use_cache=True, return_dict=True)
        past, h = out.past_key_values, out.last_hidden_state[:, -1, :]
        generated = []
        for _ in range(max_new_tokens):
            logits = self.audio_head(h).float()
            if repetition_penalty != 1.0 and generated:
                for tok in set(generated):
                    v = logits[0, tok]
                    logits[0, tok] = v / repetition_penalty if v > 0 else v * repetition_penalty
            logits = logits / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            if top_p is not None:
                s_logits, s_idx = torch.sort(logits, descending=True, dim=-1)
                cum = F.softmax(s_logits, dim=-1).cumsum(dim=-1)
                cutoff = int((cum > top_p).float().argmax(dim=-1).item())
                keep = s_idx[:, :cutoff + 1]
                filt = torch.full_like(logits, float('-inf'))
                filt.scatter_(1, keep, logits.gather(1, keep)); logits = filt
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            tok = int(next_id.item())
            if tok == self.cfg.eos_id: break
            generated.append(tok)
            mask = torch.cat([mask, torch.ones(1, 1, device=dev, dtype=torch.long)], dim=1)
            out = self.base.transformer(inputs_embeds=self.audio_emb(next_id).view(1, 1, -1),
                                        attention_mask=mask, past_key_values=past, use_cache=True, return_dict=True)
            past, h = out.past_key_values, out.last_hidden_state[:, -1, :]
        return torch.tensor(generated, dtype=torch.long, device=dev)

    @torch.no_grad()
    def generate_audio_beam(self, text_ids, beam_size=4, max_new_tokens=400, length_penalty=1.0):
        self.eval()
        c = self.cfg
        dev = text_ids.device
        beam_size = min(beam_size, c.audio_vocab_size)
        text_ids = text_ids.view(-1).long().to(dev)
        prefix = torch.cat([self.base.transformer.wte(text_ids),
                            self.audio_emb.weight[c.bos_id][None]], dim=0)   # (T+1, H)
        V, H = c.audio_vocab_size, prefix.size(1)

        def hidden_last(seqs):
            # last-position hidden state for each beam's [prefix + audio tokens] (padded batch)
            embs = [torch.cat([prefix, self.audio_emb(torch.tensor(s, device=dev, dtype=torch.long))], 0)
                    if s else prefix for s in seqs]
            lens = [e.size(0) for e in embs]
            L = max(lens)
            batch = torch.zeros(len(embs), L, H, device=dev, dtype=prefix.dtype)
            mask = torch.zeros(len(embs), L, dtype=torch.long, device=dev)
            for i, (e, l) in enumerate(zip(embs, lens)):
                batch[i, :l] = e; mask[i, :l] = 1
            hs = self.base.transformer(inputs_embeds=batch, attention_mask=mask,
                                       return_dict=True).last_hidden_state
            idx = torch.tensor([l - 1 for l in lens], device=dev)
            return hs[torch.arange(len(embs), device=dev), idx]             # (B, H)

        beam_seqs, beam_done, finished = [[]], [False], []
        beam_scores = torch.zeros(1, device=dev)
        h = hidden_last(beam_seqs)                                          # (1, H)
        for _ in range(max_new_tokens):
            log_probs = F.log_softmax(self.audio_head(h).float(), dim=-1)   # (nb, V)
            cand = beam_scores.unsqueeze(1) + log_probs
            for i in range(len(beam_seqs)):
                if beam_done[i]:
                    cand[i] = float('-inf'); cand[i, c.eos_id] = beam_scores[i]
            top_scores, top_idx = cand.view(-1).topk(min(beam_size, cand.numel()))
            beam_idx = torch.div(top_idx, V, rounding_mode='floor'); token_idx = top_idx % V
            new_seqs, new_done, new_scores = [], [], []
            for k in range(top_idx.numel()):
                src, tok = int(beam_idx[k]), int(token_idx[k])
                is_eos = tok == c.eos_id
                seq = beam_seqs[src] + ([] if (beam_done[src] or is_eos) else [tok])
                done = beam_done[src] or is_eos
                if is_eos and not beam_done[src]:
                    finished.append((top_scores[k].item() / (max(len(seq), 1) ** length_penalty), seq))
                new_seqs.append(seq); new_done.append(done); new_scores.append(top_scores[k])
            beam_seqs, beam_done, beam_scores = new_seqs, new_done, torch.stack(new_scores)
            if all(beam_done): break
            h = hidden_last(beam_seqs)
        if finished:
            best = max(finished, key=lambda x: x[0])[1]
        else:
            best = max([(s.item() / (max(len(q), 1) ** length_penalty), q)
                        for s, q in zip(beam_scores, beam_seqs)], key=lambda x: x[0])[1]
        return torch.tensor(best, dtype=torch.long, device=dev)

In [9]:
model_cfg = GPT2TTSConfig(base_model=cfg.base_model, codebook_size=CODEBOOK_SIZE)

def load_gpt2tts(ckpt_path, model_cfg):
    m = GPT2TTS(model_cfg)
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = sd.get('state_dict', sd)                                   # Lightning wraps under 'state_dict'
    inner = {k[len('model.'):]: v for k, v in sd.items() if k.startswith('model.')}
    m.load_state_dict(inner or sd, strict=True)
    return m.to(device).eval()

baseline = load_gpt2tts(CKPT_PATH, model_cfg)
print("loaded baseline:", sum(p.numel() for p in baseline.parameters())/1e6, "M params")

@torch.no_grad()
def toks_to_wav(audio_ids):
    if audio_ids.numel() == 0: return None
    return codec.toks_to_sig(audio_ids.unsqueeze(0)).squeeze(0).float().cpu().numpy().astype(np.float32)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded baseline: 137.025792 M params


## 3. Metrics — CER, UTMOS, SECS

In [10]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
asr_processor = WhisperProcessor.from_pretrained("openai/whisper-base.en")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base.en").to(device).eval()
utmos_predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(device).eval()
spk_encoder = VoiceEncoder(device=device)   # Resemblyzer d-vector speaker encoder for SECS

def _norm(s): return " ".join(str(s).lower().split())   # light norm so CER isn't dominated by case/spacing

@torch.no_grad()
def compute_cer(ref_text, wav, sr):
    if sr != 16000:
        wav = torchaudio.functional.resample(torch.from_numpy(wav), sr, 16000).numpy()
    feats = asr_processor(wav, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    hyp = asr_processor.batch_decode(asr_model.generate(feats), skip_special_tokens=True)[0]
    return jiwer.cer(_norm(ref_text), _norm(hyp)), hyp.strip()

def compute_utmos(wav, sr):
    with torch.no_grad():
        return float(utmos_predictor(torch.from_numpy(wav).float().unsqueeze(0).to(device), sr).item())

_anchor_row = train_df.iloc[(train_df['normalized_transcription'].astype(str).str.len()).idxmax()]
anchor_wav = preprocess_wav(_anchor_row['filepath'])
ANCHOR_EMB = spk_encoder.embed_utterance(anchor_wav)
print("SECS anchor:", _anchor_row['id'])

def compute_secs(wav, sr):
    if sr != 16000:
        wav = torchaudio.functional.resample(torch.from_numpy(wav), sr, 16000).numpy()
    emb = spk_encoder.embed_utterance(preprocess_wav(wav, source_sr=16000))
    return float(np.dot(emb, ANCHOR_EMB) / (np.linalg.norm(emb)*np.linalg.norm(ANCHOR_EMB)))

def score_all(ref_text, wav, sr):
    cer, hyp = compute_cer(ref_text, wav, sr)
    return {"cer": cer, "utmos": compute_utmos(wav, sr), "secs": compute_secs(wav, sr), "asr_hyp": hyp}

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Downloading: "https://github.com/tarepan/SpeechMOS/zipball/v1.2.0" to /root/.cache/torch/hub/v1.2.0.zip
Downloading: "https://github.com/tarepan/SpeechMOS/releases/download/v1.0.0/utmos22_strong_step7459_v1.pt" to /root/.cache/torch/hub/checkpoints/utmos22_strong_step7459_v1.pt


100%|██████████| 392M/392M [00:01<00:00, 212MB/s] 


Loaded the voice encoder model on cuda in 0.02 seconds.
SECS anchor: LJ038-0050


## 4. Validation sets

In [11]:
HARVARD = [
    "The birch canoe slid on the smooth planks.",
    "Glue the sheet to the dark blue background.",
    "It's easy to tell the depth of a well.",
    "These days a chicken leg is a rare dish.",
    "Rice is often served in round bowls.",
    "The juice of lemons makes fine punch.",
    "The box was thrown beside the parked truck.",
    "The hogs were fed chopped corn and garbage.",
    "Four hours of steady work faced us.",
    "A large size in stockings is hard to sell.",
    "The boy was there when the sun rose.",
    "A rod is used to catch pink salmon.",
    "The source of the huge river is the clear spring.",
    "Kick the ball straight and follow through.",
    "Help the woman get back to her feet.",
    "A pot of tea helps to pass the evening.",
    "Smoky fires lack flame and heat.",
    "The soft cushion broke the man's fall.",
    "The salt breeze came across from the sea.",
    "The girl at the booth sold fifty bonds.",
]
external_df = pd.DataFrame({"id": [f"harvard_{i:02d}" for i in range(len(HARVARD))],
                            "normalized_transcription": HARVARD, "domain": "external_harvard"})


EDGE = {
    "numbers_dates":   ["The meeting is at 3:45 PM on 07/15/2026, room 210.",
                        "It cost $1,299.99, up 12.5% from 2019."],
    "abbreviations":   ["Dr. Smith from NASA met the CEO of IBM in the USA.",
                        "See sec. 4, e.g. Fig. 2, vs. the Q3 KPI."],
    "punctuation":     ["Wait... what?! Are you — seriously — sure (about this)?",
                        "\"Stop!\" she cried; then, quietly, she left."],
    "tongue_twisters": ["She sells seashells by the seashore.",
                        "Peter Piper picked a peck of pickled peppers."],
    "rare_technical":  ["The otorhinolaryngologist calibrated the femtosecond oscilloscope.",
                        "Quinoa and acai are pseudocereal superfoods."],
    "repetition":      ["No no no no, that is not not not correct.",
                        "Very very very very long and repetitive sentence indeed."],
}
edge_rows = [{"id": f"edge_{cat}_{i}", "normalized_transcription": t, "domain": f"edge_{cat}"}
             for cat, texts in EDGE.items() for i, t in enumerate(texts)]
edge_df = pd.DataFrame(edge_rows)

print("external:", len(external_df), " edge:", len(edge_df))
wandb.log({"validation/n_external": len(external_df), "validation/n_edge": len(edge_df)})

external: 20  edge: 12


## 5. Sampling-strategy ablation

In [ ]:
SAMPLING_CONFIGS = {
    "greedy":       {"method": "sampling", "kwargs": dict(temperature=1.0, top_k=1)},
    "temp0.7":      {"method": "sampling", "kwargs": dict(temperature=0.7)},
    "topk50":       {"method": "sampling", "kwargs": dict(temperature=0.9, top_k=50)},
    "topp0.9":      {"method": "sampling", "kwargs": dict(temperature=0.9, top_p=0.9)},
    "topk50_reppen":{"method": "sampling", "kwargs": dict(temperature=0.9, top_k=50, repetition_penalty=1.2)},
    "beam4":        {"method": "beam",     "kwargs": dict(beam_size=4, length_penalty=1.0)},
}

def eval_set(model, df, set_name, n_cap=None, has_domains=True, log_audio_n=1):
    # Generate + score every SAMPLING_CONFIG over df. Returns a long-form results DataFrame.
    model.eval()
    groups = df.groupby('domain') if has_domains else [("all", df)]
    rows = []
    for sname, scfg in SAMPLING_CONFIGS.items():
        n_logged = 0
        for domain, grp in groups:
            sub = grp if n_cap is None else grp.sample(min(n_cap, len(grp)), random_state=42)
            for _, r in tqdm(sub.iterrows(), total=len(sub), desc=f"{set_name}/{sname}/{domain}", leave=False):
                tids = enc_text(r['normalized_transcription'])
                if scfg["method"] == "beam":
                    aids = model.generate_audio_beam(tids, max_new_tokens=cfg.max_new_tokens, **scfg["kwargs"])
                else:
                    aids = model.generate_audio(tids, max_new_tokens=cfg.max_new_tokens, **scfg["kwargs"])
                wav = toks_to_wav(aids)
                if wav is None: continue
                sc = score_all(r['normalized_transcription'], wav, CODEC_SR_OUT)
                wpath = cfg.out_audio / set_name / sname / f"{r['id']}.wav"
                wpath.parent.mkdir(parents=True, exist_ok=True); sf.write(wpath, wav, CODEC_SR_OUT)
                rows.append({"set": set_name, "sampling": sname, "domain": domain, "id": r['id'],
                             "text": r['normalized_transcription'], "wav_path": str(wpath), **sc})
                if n_logged < log_audio_n:
                    wandb.log({f"samples/{set_name}/{sname}": wandb.Audio(wav, sample_rate=CODEC_SR_OUT,
                               caption=f"{r['id']} cer={sc['cer']:.2f} utmos={sc['utmos']:.2f}")})
                    n_logged += 1
    return pd.DataFrame(rows)

res_test = eval_set(baseline, test_df, "test", n_cap=cfg.n_test_per_domain, has_domains=('domain' in test_df))
res_ext  = eval_set(baseline, external_df, "external", has_domains=True)
res_edge = eval_set(baseline, edge_df, "edge", has_domains=True)
ablation = pd.concat([res_test, res_ext, res_edge], ignore_index=True)
ablation.to_csv(cfg.out_audio / "ablation_results.csv", index=False)
print(len(ablation), "generated+scored rows")

In [ ]:
summary_tbl = ablation.groupby(['set', 'sampling'])[['cer', 'utmos', 'secs']].mean().round(3)

wandb.log({"ablation/summary": wandb.Table(dataframe=summary_tbl.reset_index())})
audio_tbl = wandb.Table(columns=["set","sampling","domain","id","text","asr_hyp","cer","utmos","secs","audio"])
for _, r in ablation.iterrows():
    audio_tbl.add_data(r['set'], r['sampling'], r['domain'], r['id'], r['text'], r['asr_hyp'],
                       r['cer'], r['utmos'], r['secs'], wandb.Audio(r['wav_path'], sample_rate=CODEC_SR_OUT))
wandb.log({"ablation/audio_table": audio_tbl})

comp = ablation.assign(reward=lambda d: d['utmos'] - 5*d['cer']).groupby(['set','sampling'])['reward'].mean()

## 6. RLHF via DPO — directly optimise the non-differentiable reward `UTMOS − λ·CER`

In [18]:
DPO_N_PROMPTS = 120      
DPO_K = 2
REWARD_LAMBDA = 5.0
PAIR_MARGIN = 0.15        
pref_cache = cfg.work_dir / "dpo_pairs.pt"

def reward_of(text, wav, sr):
    sc = score_all(text, wav, sr)
    return sc['utmos'] - REWARD_LAMBDA * sc['cer'], sc

@torch.no_grad()
def build_pairs(model, df, n_prompts, k):
    prompts = df.sample(min(n_prompts, len(df)), random_state=0).reset_index(drop=True)
    pairs = []
    for _, r in tqdm(prompts.iterrows(), total=len(prompts), desc="pref-gen"):
        text = r['normalized_transcription']; tids = enc_text(text)
        cands = []
        for _ in range(k):
            aids = model.generate_audio(tids, max_new_tokens=cfg.max_new_tokens, temperature=1.0, top_k=50)
            wav = toks_to_wav(aids)
            if wav is None or aids.numel() < 4: continue
            rew, _ = reward_of(text, wav, CODEC_SR_OUT)
            cands.append((rew, aids.cpu()))
        if len(cands) < 2: continue
        cands.sort(key=lambda x: -x[0])
        if cands[0][0] - cands[-1][0] < PAIR_MARGIN: continue
        pairs.append({"text_ids": tids.cpu(), "chosen": cands[0][1], "rejected": cands[-1][1],
                      "margin": cands[0][0] - cands[-1][0]})
    return pairs

if pref_cache.exists():
    dpo_pairs = torch.load(pref_cache, weights_only=False); print("loaded cached pairs")
else:
    dpo_pairs = build_pairs(baseline, train_df, DPO_N_PROMPTS, DPO_K)
    torch.save(dpo_pairs, pref_cache)
print("preference pairs:", len(dpo_pairs))
wandb.log({"dpo/n_pairs": len(dpo_pairs),
           "dpo/mean_margin": float(np.mean([p['margin'] for p in dpo_pairs])) if dpo_pairs else 0.0})

pref-gen:   0%|          | 0/120 [00:00<?, ?it/s]

preference pairs: 95


In [19]:
def seq_logprob(model, text_ids, audio_ids):
    c = model.cfg
    wte = model.base.transformer.wte
    dev = text_ids.device
    a_emb = model.audio_emb(audio_ids)
    seq = torch.cat([wte(text_ids), model.audio_emb.weight[c.bos_id][None], a_emb,
                     model.audio_emb.weight[c.eos_id][None]], 0).unsqueeze(0)
    h = model.base.transformer(inputs_embeds=seq, return_dict=True).last_hidden_state
    logp = F.log_softmax(model.audio_head(h).float(), dim=-1)[0]
    tl, al = text_ids.numel(), audio_ids.numel()
    targets = torch.cat([audio_ids, torch.tensor([c.eos_id], device=dev)]) 
    pred_pos = torch.arange(tl, tl + al + 1, device=dev)                       
    return logp[pred_pos, targets].sum()

policy = load_gpt2tts(CKPT_PATH, model_cfg).train()
ref = load_gpt2tts(CKPT_PATH, model_cfg).eval()
for p in ref.parameters(): p.requires_grad_(False)

BETA, DPO_LR, DPO_EPOCHS, ACCUM = 0.1, 1e-6, 2, 8
opt = torch.optim.AdamW(policy.parameters(), lr=DPO_LR)

step = 0
for epoch in range(DPO_EPOCHS):
    random.shuffle(dpo_pairs)
    opt.zero_grad()
    running = {"loss": [], "acc": [], "margin": []}
    for j, ex in enumerate(tqdm(dpo_pairs, desc=f"dpo epoch {epoch}")):
        t = ex['text_ids'].to(device); ch = ex['chosen'].to(device); rj = ex['rejected'].to(device)
        pol_c, pol_r = seq_logprob(policy, t, ch), seq_logprob(policy, t, rj)
        with torch.no_grad():
            ref_c, ref_r = seq_logprob(ref, t, ch), seq_logprob(ref, t, rj)
        margin = (pol_c - ref_c) - (pol_r - ref_r)
        loss = -F.logsigmoid(BETA * margin)
        (loss / ACCUM).backward()
        running["loss"].append(loss.item())
        running["acc"].append(float(margin.item() > 0))
        running["margin"].append(margin.item())
        if (j + 1) % ACCUM == 0 or j + 1 == len(dpo_pairs):
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            opt.step(); opt.zero_grad(); step += 1
            if step % 5 == 0:
                wandb.log({"dpo/loss": np.mean(running["loss"][-ACCUM:]),
                           "dpo/pref_acc": np.mean(running["acc"][-50:]),
                           "dpo/reward_margin": np.mean(running["margin"][-ACCUM:]), "dpo/step": step})
    print(f"epoch {epoch}: loss={np.mean(running['loss']):.4f}  pref_acc={np.mean(running['acc']):.3f}")

policy.eval()
dpo_ckpt = cfg.work_dir / "gpt2tts_dpo.pt"
torch.save(policy.state_dict(), dpo_ckpt)
print("saved DPO policy ->", dpo_ckpt)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dpo epoch 0:   0%|          | 0/95 [00:00<?, ?it/s]

epoch 0: loss=1.2992  pref_acc=0.526


dpo epoch 1:   0%|          | 0/95 [00:00<?, ?it/s]

epoch 1: loss=1.0854  pref_acc=0.589
saved DPO policy -> /kaggle/working/gpt2tts_dpo.pt


In [ ]:
DPO_DECODE = {"topk50": {"method": "sampling", "kwargs": dict(temperature=0.9, top_k=50)}}
_saved_configs = SAMPLING_CONFIGS
SAMPLING_CONFIGS = DPO_DECODE
try:
    def eval_models(models, name):
        out = []
        for tag, m in models.items():
            for setname, d, cap, dom in [("test", test_df, cfg.n_test_per_domain, 'domain' in test_df),
                                         ("external", external_df, None, True),
                                         ("edge", edge_df, None, True)]:
                r = eval_set(m, d, f"{name}_{tag}_{setname}", n_cap=cap, has_domains=dom, log_audio_n=0)
                r["model"] = tag; out.append(r)
        return pd.concat(out, ignore_index=True)
    dpo_cmp = eval_models({"baseline": baseline, "dpo": policy}, "dpocmp")
finally:
    SAMPLING_CONFIGS = _saved_configs

cmp_tbl = dpo_cmp.assign(set=lambda d: d['set'].str.replace(r'^dpocmp_(baseline|dpo)_', '', regex=True)) \
                 .groupby(['model', 'set'])[['cer', 'utmos', 'secs']].mean().round(3)
wandb.log({"dpo/comparison": wandb.Table(dataframe=cmp_tbl.reset_index())})

delta = (cmp_tbl.xs('dpo', level='model') - cmp_tbl.xs('baseline', level='model')).round(3)
wandb.log({"dpo/delta": wandb.Table(dataframe=delta.reset_index())})

In [21]:
wandb.finish()

dpo/loss,▃▁█▆
dpo/mean_margin,▁
dpo/n_pairs,▁
dpo/pref_acc,▁▁█▁
dpo/reward_margin,▅█▁▄
dpo/step,▁▃▆█
validation/n_edge,▁
validation/n_external,▁
dpo/loss,1.34029
dpo/mean_margin,2.70245
dpo/n_pairs,95
